# Day 10: RAG + Memory Inside LangGraph 🔗

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. ChromaDB knowledge base — load and embed documents
2. RAG-only graph — retrieve → answer, no memory
3. Memory-only chatbot — multi-turn memory, no retrieval
4. Full RAG + Memory graph — all four nodes together
5. Add a query router — agent decides: retrieve or use memory

**Time:** ~2 hours | **Prerequisites:** Days 3, 4-5, 8 complete

---

## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# LOCAL users: SKIP this cell
# ============================================================
# !pip install langgraph langchain-groq langchain-core langchain-community \
#              langchain-chroma langchain-huggingface sentence-transformers \
#              chromadb python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
from importlib.metadata import version

print(f"LangGraph version: {version('langgraph')}")

# One LLM used throughout the entire notebook
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = llm.invoke("Say 'Day 10 ready!' in exactly 3 words.")
print(response.content)

---
## 1. 📚 Build the ChromaDB Knowledge Base

Same pattern as Days 4-5. We'll load 5 short documents about Agentic AI topics, chunk them, embed them, and store them in ChromaDB.

This knowledge base is what the agent will query for the rest of today.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# ── Embedding model (same as Days 4-5) ───────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded")

# ── Knowledge base documents ──────────────────────────────
# In a real project: load PDFs, web pages, or any documents
# Here we use 5 short paragraphs about Agentic AI topics
documents = [
    {
        "id": "doc_001",
        "topic": "LangGraph Overview",
        "text": """LangGraph is an open-source library built by LangChain Inc. for creating stateful,
multi-actor applications using large language models. It extends LangChain by modeling
agent workflows as directed graphs where nodes represent processing steps and edges
represent transitions. LangGraph supports conditional routing, loops, and human-in-the-loop
workflows. It uses a shared State dictionary to pass information between nodes. The
MemorySaver checkpointer enables persistent state across multiple invocations using
thread IDs to identify conversations."""
    },
    {
        "id": "doc_002",
        "topic": "RAG Architecture",
        "text": """Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses
by retrieving relevant documents from a knowledge base before generating an answer.
The pipeline has four stages: document loading, chunking, embedding, and retrieval.
Documents are split into chunks of 500-1000 characters with overlap to preserve context.
Each chunk is converted into a vector embedding using models like all-MiniLM-L6-v2.
At query time, the question is embedded and the top-k most similar chunks are retrieved
using cosine similarity. These chunks form the context for the LLM's response."""
    },
    {
        "id": "doc_003",
        "topic": "Agent Memory Systems",
        "text": """AI agents require memory to maintain conversation context across multiple turns.
Short-term memory stores the current conversation as a list of message dictionaries,
each with a role (user or assistant) and content field. A sliding window approach
limits memory to the last N messages to control token costs. Long-term memory uses
vector databases like ChromaDB to store and retrieve information across sessions.
LangGraph's MemorySaver checkpointer provides built-in state persistence using thread
IDs, enabling agents to resume conversations from exactly where they left off."""
    },
    {
        "id": "doc_004",
        "topic": "Self-Reflection Pattern",
        "text": """Self-reflection is an agent design pattern where the LLM evaluates and improves
its own output. The pattern has three phases: generate an initial answer, critique it
by asking what is missing or incorrect, and improve the answer based on the critique.
This works because generating and evaluating are different cognitive tasks — the
critique prompt puts the LLM in 'fresh eyes' mode. In LangGraph, self-reflection is
implemented as a loop: generate_node → critique_node → improve_node → conditional
edge that either loops back or exits. A maximum iteration limit (usually 3) is always
required to prevent infinite loops."""
    },
    {
        "id": "doc_005",
        "topic": "Groq API and LLM Providers",
        "text": """Groq is an AI infrastructure company that provides fast LLM inference using
custom LPU (Language Processing Unit) hardware. The Groq API is compatible with the
OpenAI API format, meaning the same JSON structure works across providers. The free
tier provides approximately 100,000 tokens per day with a rate limit of 30 requests
per minute. The llama-3.3-70b-versatile model on Groq delivers over 300 tokens per
second — significantly faster than cloud GPU inference. In LangChain, ChatGroq is
the connector class that reads the GROQ_API_KEY from the environment automatically."""
    },
]

print(f"Prepared {len(documents)} documents")
for doc in documents:
    print(f"  • {doc['topic']} ({len(doc['text'].split())} words)")

In [ ]:
# ── Create ChromaDB collection ────────────────────────────
client = chromadb.Client()  # in-memory for this notebook

# Delete if exists (so we can re-run this cell)
try:
    client.delete_collection("agentic_ai_kb")
except:
    pass

collection = client.create_collection("agentic_ai_kb")

# ── Embed and store all documents ─────────────────────────
texts = [doc["text"] for doc in documents]
ids   = [doc["id"]   for doc in documents]

print("Embedding documents...")
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": doc["topic"]} for doc in documents]
)

print(f"✅ {collection.count()} documents stored in ChromaDB")

# ── Quick retrieval test ───────────────────────────────────
test_query = "How does LangGraph handle state?"
test_embedding = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=test_embedding, n_results=2)
print(f"\nTest query: '{test_query}'")
print(f"Top result topic: {results['metadatas'][0][0]['topic']}")
print(f"Top result preview: {results['documents'][0][0][:150]}...")

**Knowledge base is ready.** 5 documents embedded and stored. Now let's build the LangGraph workflows that use it.

---
## 2. RAG-Only Graph — No Memory

The simplest version first. Three nodes:
```
[retrieve] → [answer] → END
```
One question, one answer, no conversation history. This is the baseline.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── State ──────────────────────────────────────────────────
class RAGState(TypedDict):
    question:  str
    retrieved: str   # chunks from ChromaDB
    answer:    str


# ── Node 1: Retrieve ───────────────────────────────────────
def retrieve_node(state: RAGState) -> dict:
    """Queries ChromaDB and returns top 3 relevant chunks."""
    question = state["question"]

    # Embed the question and search
    q_embedding = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=3)

    # Combine chunks into one context string
    chunks = results["documents"][0]
    topics = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(
        f"[Source: {topics[i]}]\n{chunks[i]}" for i in range(len(chunks))
    )

    print(f"  [retrieve] Found {len(chunks)} chunks for: '{question[:50]}'")
    print(f"  [retrieve] Sources: {topics}")
    return {"retrieved": context}


# ── Node 2: Answer ─────────────────────────────────────────
def answer_node(state: RAGState) -> dict:
    """Uses the retrieved context to answer the question."""
    question  = state["question"]
    retrieved = state["retrieved"]

    messages = [
        SystemMessage(content=f"""You are a helpful assistant for an Agentic AI course.
Answer the question using ONLY the context below.
If the answer is not in the context, say: 'I don't have that information in my knowledge base.'

CONTEXT:
{retrieved}"""),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    print(f"  [answer] Response generated")
    return {"answer": response.content}


# ── Build the graph ────────────────────────────────────────
rag_graph = StateGraph(RAGState)
rag_graph.add_node("retrieve", retrieve_node)
rag_graph.add_node("answer",   answer_node)

rag_graph.set_entry_point("retrieve")
rag_graph.add_edge("retrieve", "answer")
rag_graph.add_edge("answer",   END)

rag_app = rag_graph.compile()
print("RAG-only graph compiled!")

In [ ]:
# Test 1 — question that should be in the knowledge base
print("=" * 55)
result = rag_app.invoke({"question": "What is the sliding window approach in agent memory?"})
print(f"\nAnswer:\n{result['answer']}")

In [ ]:
# Test 2 — question NOT in the knowledge base
print("=" * 55)
result2 = rag_app.invoke({"question": "What is the price of a GPU on Amazon?"})
print(f"\nAnswer:\n{result2['answer']}")
# Should say: I don't have that information in my knowledge base

**Limitation spotted:** Ask the same agent two questions back to back — it has no idea what you asked in Question 1 when you ask Question 2. Each call starts fresh. That's what Section 3 fixes.

---
## 3. Memory-Only Chatbot — No Retrieval

Now build the other half: a conversational chatbot that **remembers** across turns using `MemorySaver` and `thread_id`. No document retrieval — just memory.

```
[chat_node] → END
```
Simple graph, but with `MemorySaver` the state persists between `invoke()` calls.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── State ──────────────────────────────────────────────────
class MemoryState(TypedDict):
    question: str
    messages: List[dict]   # conversation history
    answer:   str


# ── Node: Chat with memory ─────────────────────────────────
def chat_node(state: MemoryState) -> dict:
    """Reads history, adds current question, calls LLM, saves response."""
    question = state["question"]
    messages = state.get("messages", [])

    # Add current question to history
    messages = messages + [{"role": "user", "content": question}]

    # Sliding window: keep only last 6 messages (3 turns)
    if len(messages) > 6:
        messages = messages[-6:]
        print(f"  [chat] Sliding window applied — keeping last 6 messages")

    # Build LangChain messages
    lc_messages = [SystemMessage(content="You are a helpful Agentic AI course assistant. Be concise.")]
    for msg in messages:
        if msg["role"] == "user":
            lc_messages.append(HumanMessage(content=msg["content"]))
        else:
            lc_messages.append(AIMessage(content=msg["content"]))

    response = llm.invoke(lc_messages)
    answer   = response.content

    # Save response to history
    messages = messages + [{"role": "assistant", "content": answer}]

    print(f"  [chat] Turn complete. History length: {len(messages)} messages")
    return {"messages": messages, "answer": answer}


# ── Build the graph WITH MemorySaver ──────────────────────
mem_graph = StateGraph(MemoryState)
mem_graph.add_node("chat", chat_node)
mem_graph.set_entry_point("chat")
mem_graph.add_edge("chat", END)

checkpointer = MemorySaver()
mem_app = mem_graph.compile(checkpointer=checkpointer)

print("Memory chatbot compiled with MemorySaver!")

In [ ]:
# thread_id identifies the conversation — same ID = same conversation
config = {"configurable": {"thread_id": "student-001"}}

def chat(question):
    """Helper to send a message and print the response."""
    result = mem_app.invoke({"question": question}, config=config)
    print(f"You:  {question}")
    print(f"Bot:  {result['answer']}")
    print()
    return result

# Multi-turn conversation — agent should remember across turns
print("=" * 55)
print("CONVERSATION — thread_id: student-001")
print("=" * 55)
chat("Hi! My name is Priya and I'm learning LangGraph.")
chat("What is the main concept in LangGraph?")
chat("What is my name?")   # Should remember "Priya" from Turn 1

In [ ]:
# New thread_id = fresh conversation — no memory of Priya
config2 = {"configurable": {"thread_id": "student-002"}}

def chat2(question):
    result = mem_app.invoke({"question": question}, config=config2)
    print(f"You:  {question}")
    print(f"Bot:  {result['answer']}")
    print()

print("=" * 55)
print("NEW CONVERSATION — thread_id: student-002")
print("=" * 55)
chat2("What is my name?")   # Should say: I don't know your name
chat2("I told you earlier — my name is Priya!")   # This thread has no history

**Key observations:**
- `thread_id: student-001` remembered "Priya" across turns automatically
- `thread_id: student-002` started completely fresh — no knowledge of thread 001
- The `messages` list in State grew with each turn — that's the memory
- `MemorySaver` handled saving/restoring State between `invoke()` calls

---
## 4. Full RAG + Memory Graph — All Four Nodes

Now combine both. Four nodes:
```
[memory_node] → [retrieve_node] → [answer_node] → [save_node] → END
```
Multi-turn conversation grounded in the knowledge base.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── State — carries everything ─────────────────────────────
class RAGMemoryState(TypedDict):
    question:  str
    messages:  List[dict]   # full conversation history
    retrieved: str          # chunks from ChromaDB
    answer:    str


# ── Node 1: Memory — add question, apply sliding window ───
def memory_node(state: RAGMemoryState) -> dict:
    """Adds the new question to conversation history."""
    question = state["question"]
    messages = state.get("messages", [])

    messages = messages + [{"role": "user", "content": question}]

    # Sliding window: keep last 6 messages
    if len(messages) > 6:
        messages = messages[-6:]

    print(f"  [memory] History: {len(messages)} messages")
    return {"messages": messages}


# ── Node 2: Retrieve — query ChromaDB ─────────────────────
def retrieval_node(state: RAGMemoryState) -> dict:
    """Finds the most relevant document chunks for the question."""
    question = state["question"]

    q_embedding = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=3)

    chunks = results["documents"][0]
    topics = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(
        f"[Source: {topics[i]}]\n{chunks[i]}" for i in range(len(chunks))
    )

    print(f"  [retrieve] Sources: {topics}")
    return {"retrieved": context}


# ── Node 3: Answer — memory + RAG + question → LLM ───────
def answer_node_full(state: RAGMemoryState) -> dict:
    """LLM sees: retrieved context + conversation history + current question."""
    question  = state["question"]
    retrieved = state["retrieved"]
    messages  = state.get("messages", [])

    # System prompt includes the retrieved context
    system_prompt = f"""You are a helpful assistant for an Agentic AI course.
Answer using the knowledge base context below.
If the answer is not in the context, use your general knowledge but say so.
You also have access to the conversation history — use it to maintain continuity.

KNOWLEDGE BASE CONTEXT:
{retrieved}"""

    # Build messages: system + history (except last user msg) + current question
    lc_messages = [SystemMessage(content=system_prompt)]

    # Add conversation history (all except the last message, which is the current question)
    for msg in messages[:-1]:
        if msg["role"] == "user":
            lc_messages.append(HumanMessage(content=msg["content"]))
        else:
            lc_messages.append(AIMessage(content=msg["content"]))

    # Add current question
    lc_messages.append(HumanMessage(content=question))

    response = llm.invoke(lc_messages)
    print(f"  [answer] Generated ({len(response.content)} chars)")
    return {"answer": response.content}


# ── Node 4: Save — append answer to history ───────────────
def save_node(state: RAGMemoryState) -> dict:
    """Saves the assistant's answer back into conversation history."""
    messages = state.get("messages", [])
    answer   = state["answer"]
    messages = messages + [{"role": "assistant", "content": answer}]
    print(f"  [save] Answer saved. History now: {len(messages)} messages")
    return {"messages": messages}


# ── Build the graph ────────────────────────────────────────
rag_mem_graph = StateGraph(RAGMemoryState)

rag_mem_graph.add_node("memory",   memory_node)
rag_mem_graph.add_node("retrieve", retrieval_node)
rag_mem_graph.add_node("answer",   answer_node_full)
rag_mem_graph.add_node("save",     save_node)

rag_mem_graph.set_entry_point("memory")
rag_mem_graph.add_edge("memory",   "retrieve")
rag_mem_graph.add_edge("retrieve", "answer")
rag_mem_graph.add_edge("answer",   "save")
rag_mem_graph.add_edge("save",     END)

# MemorySaver persists state across invoke() calls
checkpointer = MemorySaver()
rag_mem_app = rag_mem_graph.compile(checkpointer=checkpointer)

print("Full RAG + Memory graph compiled!")

In [ ]:
# Run a multi-turn conversation with document grounding
config = {"configurable": {"thread_id": "rag-session-001"}}

def ask(question):
    print(f"\nYou: {question}")
    result = rag_mem_app.invoke({"question": question}, config=config)
    print(f"Bot: {result['answer']}")
    return result

print("=" * 55)
print("Multi-turn RAG conversation")
print("=" * 55)

# Turn 1: question from knowledge base
ask("What is RAG and how does it work?")

In [ ]:
# Turn 2: follow-up that references Turn 1 — tests memory
ask("You mentioned embeddings. What model is used for them?")

In [ ]:
# Turn 3: question about a DIFFERENT document topic
ask("Now tell me about the self-reflection pattern.")

In [ ]:
# Turn 4: memory test — does it remember Turn 1?
ask("What was the first topic we discussed?")

**What just happened:**
- Turn 1: Retrieved RAG chunks, answered from knowledge base
- Turn 2: Used "You mentioned embeddings" — agent knows what it said in Turn 1
- Turn 3: Retrieved different chunks for self-reflection
- Turn 4: Looked at conversation history to answer "first topic"

The agent has both document knowledge AND conversation memory simultaneously.

---
## 5. Query Router — Smart Retrieval

Not every question needs ChromaDB. If someone asks "what did you just say?", retrieving document chunks is wasteful and may confuse the answer. The router decides:

```
[memory] → [router] → retrieve → [answer] → [save]
                    ↘ memory_only → [answer] → [save]
```

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── State — adds a route field ─────────────────────────────
class RouterRAGState(TypedDict):
    question:  str
    messages:  List[dict]
    route:     str          # "retrieve" or "memory_only"
    retrieved: str
    answer:    str


# ── Node 1: Memory (same as before) ───────────────────────
def router_memory_node(state: RouterRAGState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    messages = messages + [{"role": "user", "content": question}]
    if len(messages) > 6:
        messages = messages[-6:]
    return {"messages": messages}


# ── Node 2: Router — decide retrieve or memory_only ───────
def router_node(state: RouterRAGState) -> dict:
    """LLM decides if retrieval is needed for this question."""
    question = state["question"]
    messages = state.get("messages", [])

    # Build recent history summary for context
    recent = messages[-4:] if len(messages) > 4 else messages
    history_summary = "; ".join(
        f"{m['role']}: {m['content'][:60]}" for m in recent[:-1]  # exclude current question
    ) or "No history yet"

    prompt = f"""You are a router for a RAG chatbot with a knowledge base about Agentic AI.

Recent conversation: {history_summary}
Current question: {question}

Does this question require searching the knowledge base (documents about LangGraph, RAG,
memory systems, self-reflection, Groq API)?

Or can it be answered from the conversation history alone?

Reply with ONLY one word: 'retrieve' or 'memory_only'"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:
        decision = "memory_only"
    else:
        decision = "retrieve"

    print(f"  [router] '{question[:50]}' → {decision}")
    return {"route": decision}


# ── Node 3a: Retrieve (only if route = 'retrieve') ────────
def smart_retrieve_node(state: RouterRAGState) -> dict:
    question    = state["question"]
    q_embedding = embedder.encode([question]).tolist()
    results     = collection.query(query_embeddings=q_embedding, n_results=3)
    chunks      = results["documents"][0]
    topics      = [m["topic"] for m in results["metadatas"][0]]
    context     = "\n\n---\n\n".join(
        f"[Source: {topics[i]}]\n{chunks[i]}" for i in range(len(chunks))
    )
    print(f"  [retrieve] Sources: {topics}")
    return {"retrieved": context}


# ── Node 3b: Skip retrieval (route = 'memory_only') ───────
def skip_retrieve_node(state: RouterRAGState) -> dict:
    print(f"  [skip_retrieve] Using memory only — no ChromaDB search")
    return {"retrieved": ""}   # empty — answer node handles this


# ── Routing function ───────────────────────────────────────
def route_decision(state: RouterRAGState) -> str:
    return state["route"]  # "retrieve" or "memory_only"


# ── Node 4: Answer — works with or without retrieval ──────
def smart_answer_node(state: RouterRAGState) -> dict:
    question  = state["question"]
    retrieved = state.get("retrieved", "")
    messages  = state.get("messages", [])

    # Adapt system prompt based on whether we have retrieved content
    if retrieved:
        sys_content = f"""You are a helpful Agentic AI course assistant.
Use the knowledge base context below to answer.
Also use conversation history for continuity.

KNOWLEDGE BASE:
{retrieved}"""
    else:
        sys_content = """You are a helpful Agentic AI course assistant.
Answer based on the conversation history."""

    lc_messages = [SystemMessage(content=sys_content)]
    for msg in messages[:-1]:
        if msg["role"] == "user":
            lc_messages.append(HumanMessage(content=msg["content"]))
        else:
            lc_messages.append(AIMessage(content=msg["content"]))
    lc_messages.append(HumanMessage(content=question))

    response = llm.invoke(lc_messages)
    print(f"  [answer] Done (used {'retrieval' if retrieved else 'memory only'})")
    return {"answer": response.content}


# ── Node 5: Save ───────────────────────────────────────────
def smart_save_node(state: RouterRAGState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


# ── Build the graph ────────────────────────────────────────
router_graph = StateGraph(RouterRAGState)

router_graph.add_node("memory",       router_memory_node)
router_graph.add_node("router",       router_node)
router_graph.add_node("retrieve",     smart_retrieve_node)
router_graph.add_node("skip_retrieve",skip_retrieve_node)
router_graph.add_node("answer",       smart_answer_node)
router_graph.add_node("save",         smart_save_node)

router_graph.set_entry_point("memory")
router_graph.add_edge("memory", "router")

router_graph.add_conditional_edges(
    "router",
    route_decision,
    {"retrieve": "retrieve", "memory_only": "skip_retrieve"}
)

router_graph.add_edge("retrieve",      "answer")
router_graph.add_edge("skip_retrieve", "answer")
router_graph.add_edge("answer",        "save")
router_graph.add_edge("save",          END)

router_checkpointer = MemorySaver()
router_app = router_graph.compile(checkpointer=router_checkpointer)

print("Router + RAG + Memory graph compiled!")

In [ ]:
config = {"configurable": {"thread_id": "smart-session-001"}}

def smart_ask(question):
    print(f"\nYou: {question}")
    result = router_app.invoke({"question": question}, config=config)
    print(f"Bot: {result['answer']}")
    return result

print("=" * 55)
print("SMART RAG + Memory with Query Router")
print("=" * 55)

# Should route to RETRIEVE — needs knowledge base
smart_ask("What is the self-reflection pattern in LangGraph?")

In [ ]:
# Should route to MEMORY ONLY — can be answered from conversation history
smart_ask("Can you summarise what you just told me in one sentence?")

In [ ]:
# Should route to RETRIEVE — new topic from knowledge base
smart_ask("Now explain how Groq's API differs from OpenAI's API.")

In [ ]:
# Should route to MEMORY ONLY — asking about conversation history
smart_ask("What was the first question I asked you?")

---
## 6. 🎯 Your Turn — Challenge Exercises

### Challenge 1 (Easy): Expand the knowledge base
Add 2 more documents to the ChromaDB collection from Section 1 — pick any topics relevant to the course (e.g., CrewAI, Plan-Execute pattern, ReWOO). Then test that the RAG graph retrieves them correctly.

In [ ]:
# YOUR CODE HERE
# Hint:
# new_docs = [{"id": "doc_006", "topic": "CrewAI", "text": "..."}]
# new_texts = [d["text"] for d in new_docs]
# new_embeddings = embedder.encode(new_texts).tolist()
# collection.add(documents=new_texts, embeddings=new_embeddings,
#                ids=[d["id"] for d in new_docs],
#                metadatas=[{"topic": d["topic"]} for d in new_docs])
# print(f"Collection now has {collection.count()} documents")

### Challenge 2 (Medium): Show the sources
Modify `answer_node_full` in Section 4 to also return which source documents were used. The answer should end with:
```
Sources used: LangGraph Overview, Agent Memory Systems
```

In [ ]:
# YOUR CODE HERE
# Hint: In retrieve_node, also return {"sources": topics}
# Add sources: List[str] to the State TypedDict
# In answer_node, append "\n\nSources used: " + ", ".join(state['sources']) to the answer

### Challenge 3 (Hard): Persistent storage
Replace the in-memory `chromadb.Client()` with a persistent client that saves to disk:
```python
client = chromadb.PersistentClient(path="./chroma_db")
```
Test that your knowledge base survives a kernel restart — documents should still be there when you reconnect.

In [ ]:
# YOUR CODE HERE
# Hint: Replace chromadb.Client() with chromadb.PersistentClient(path="./chroma_db")
# The rest of the code stays identical
# After restarting kernel, try: client.get_collection("agentic_ai_kb").count()
# You should see 5 (or more if you did Challenge 1)

---
## 7. Day 10 Wrap-up

### What you built today:
- ✅ **ChromaDB knowledge base** — 5 documents embedded and stored
- ✅ **RAG-only graph** — retrieve → answer, no memory
- ✅ **Memory-only chatbot** — MemorySaver + thread_id, no retrieval
- ✅ **Full RAG + Memory** — all 4 nodes, multi-turn + document grounding
- ✅ **Query router** — smart routing: retrieve vs memory_only per question

### The three rules for RAG + Memory:
```python
# Rule 1: Sliding window keeps cost bounded
if len(messages) > 6:
    messages = messages[-6:]

# Rule 2: Same thread_id = same conversation
config = {"configurable": {"thread_id": "unique-id"}}

# Rule 3: Retrieved context goes in the SYSTEM message, not the user message
SystemMessage(content=f"CONTEXT:\n{retrieved}")
```

### Tomorrow — Day 11: Agent Evaluation & Red-Teaming
How do you know if your agent is actually good? Faithfulness scores, answer relevance, context precision — measuring and improving agent quality.

---
### 📚 Resources:
- [LangGraph memory how-to](https://langchain-ai.github.io/langgraph/how-tos/memory/)
- [ChromaDB docs](https://docs.trychroma.com/)
- [RAGAS evaluation](https://docs.ragas.io/) — used on Day 11